# FT listwise — BIEN THE AM HANG 2-10  (chay 10/09)

**Chi doi MOT bien** so voi ban da nop 0,711: tap huan luyen doi tu am hang 10-20
sang **am hang 2-10** (`trainset_listwise_hard_clean.jsonl`, 5507 nhom, md5 f51f73ef).
Giu nguyen loss, LR=1e-5, EPOCHS=1, HOLDOUT=400, MAXLEN=1024, GROUPS=1, ACC=16.

**Ly do:** 67% cau sai co gold o hang 2-3, nhung model dang hoc tren am hang 10-20 —
mot bien quyet dinh no khong bao gio phai doi mat. Val acc TRUOC huan luyen tren bo
de da la 0,83 = bai qua de.

## NGUONG DA KHOA — doc TRUOC khi xem ket qua
Do tren dev300 bang dung luoi `richharness_run.ipynb`.
**NHAN** neu: thang ban FT hien tai o **>= 15/20 o** VA McNemar tai N=3,k=10 cho **p < 0.05**.
**KHONG DAT => DONG, khong nop.** Khong duoc doi nguong sau khi thay so.

## Da sua 10-09: guard 500 buoc gate bang VAL ACC, khong bang loss
Lan chay dau bi dung oan o buoc 500 (`d=-0.1208`) vi cua so do nam TRON trong LR warmup
(ACC=16 -> 31 buoc toi uu; warmup = 32 buoc). Val acc luc do da tang 0.7300 -> 0.7650.
Bay gio chi dung neu **val acc khong tang**. Chay het epoch ~48 phut.


In [ ]:
import os, sys, json, time, random, math, hashlib
os.environ["PYTORCH_CUDA_ALLOC_CONF"]="expandable_segments:True"   # PHAI dat TRUOC import torch
import numpy as np, torch, torch.nn.functional as Fn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification

ROOT="/kaggle/input"
def find(name):
    for r,_,fs in os.walk(ROOT):
        if name in fs: return os.path.join(r,name)
    raise FileNotFoundError(f"KHONG THAY {name} duoi {ROOT}")
TRAIN  = find("trainset_listwise_hard_clean.jsonl")   # AM HANG 2-10, da loc nhan tu mau thuan
# quy tac 8: nhan dien bang RUOT, khong bang TEN.
SIG_TRAIN_MD5 = "f51f73efdba8f9973ef1ee18fc6693dc"
SIG_TRAIN_N   = 5507
_h=hashlib.md5(open(TRAIN,"rb").read()).hexdigest()
_n=sum(1 for _ in open(TRAIN,encoding="utf-8"))
print(f"  tap train md5={_h}  nhom={_n}")
assert _h==SIG_TRAIN_MD5, f"SAI TAP TRAIN.\n  co  : {_h}\n  can : {SIG_TRAIN_MD5} (trainset_listwise_hard_clean)"
assert _n==SIG_TRAIN_N, f"so nhom {_n} != {SIG_TRAIN_N}"
DEVF   = find("dev_300_locked.json")                      # o 3 can -> giai quyet NGAY BAY GIO
SCORESF= find("scores_dev300_fusion_M20_K20.json")        # o 3 can
UTIL   = os.path.dirname(find("deep_chunk.py"))
sys.path.insert(0, UTIL)                                   # PHAI truoc import deep_chunk
import deep_chunk as DC                                    # keo theo make_candidates_fallback
DC.MERGE_CHARS = 1800
CTX = None
for _r,_,_fs in os.walk(ROOT):
    if any(f.startswith("context_") and f.endswith(".json") for f in _fs): CTX=_r; break
assert CTX, f"KHONG THAY thu muc context_*.json duoi {ROOT}"
_t = DC.read_passage(CTX, os.listdir(CTX)[0].replace("context_","").replace(".json",""))
assert len(_t) > 50, "read_passage tra ve rong -> CTX sai thu muc"
for _n,_v in [("TRAIN",TRAIN),("DEV",DEVF),("SCORES",SCORESF),("UTIL",UTIL),("CTX",CTX)]:
    print(f"  {_n:<7} {_v}")
print(f"  kho {sum(1 for f in os.listdir(CTX) if f.startswith('context_')):,} van ban · read_passage OK\n")

# 10-09: lo tai lap da phat hien — ban 0,711 KHONG seed torch/CUDA nen train lai ra model khac.
torch.manual_seed(0); torch.cuda.manual_seed_all(0); random.seed(0); np.random.seed(0)
torch.use_deterministic_algorithms(True, warn_only=True)

MODEL   = "AITeamVN/Vietnamese_Reranker"
MAXLEN  = 1024          # = max_length luc cham that
GROUPS  = 1             # 1 nhom = 5 cap. GROUPS=2 -> OOM tren T4 (da do 08/09)
ACC     = 16            # batch hieu dung van 16 nhom
LR      = 1e-5
EPOCHS  = 1
HOLDOUT = 400

rows=[json.loads(l) for l in open(TRAIN,encoding="utf-8")]
random.Random(0).shuffle(rows)
val, tr = rows[:HOLDOUT], rows[HOLDOUT:]
print(f"{len(tr)} nhom huan luyen · {len(val)} nhom kiem tra")

tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=1).cuda()
model.gradient_checkpointing_enable()      # ~10x it activation, doi ~35% toc do. KHONG CO -> OOM
model.config.use_cache = False             # bat buoc khi bat gradient checkpointing
print(f"tham so: {sum(p.numel() for p in model.parameters())/1e9:.3f}B · gradient checkpointing: BAT")
print(f"VRAM sau khi nap model: {torch.cuda.memory_allocated()/2**30:.2f} GB / {torch.cuda.get_device_properties(0).total_memory/2**30:.2f} GB")

class DS(Dataset):
    def __init__(s,r): s.r=r
    def __len__(s): return len(s.r)
    def __getitem__(s,i):
        x=s.r[i]; return x["question"], [x["pos"]]+x["negs"]

def collate(b):
    qs=[q for q,ds in b for _ in ds]; ts=[d for _,ds in b for d in ds]
    e=tok(qs, ts, truncation=True, max_length=MAXLEN, padding=True, return_tensors="pt")
    return {k:v.cuda() for k,v in e.items()}, len(b), len(b[0][1])

In [ ]:
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
scaler = torch.amp.GradScaler("cuda")
dl = DataLoader(DS(tr), batch_size=GROUPS, shuffle=True, collate_fn=collate, drop_last=True)
total = len(dl)*EPOCHS
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=total//ACC+1, pct_start=0.1)

@torch.no_grad()
def val_acc():
    model.eval(); ok=0
    for i in range(0,len(val),GROUPS):
        b=[(x["question"],[x["pos"]]+x["negs"]) for x in val[i:i+GROUPS]]
        if not b: break
        enc,g,k=collate(b)
        with torch.autocast("cuda",dtype=torch.float16):
            lg=model(**enc).logits.view(g,k)
        ok+=int((lg.argmax(1)==0).sum())
    model.train(); return ok/len(val)

v0 = val_acc()
print(f"do chinh xac kiem tra TRUOC khi huan luyen: {v0:.4f}  (ngau nhien = 0.20)")
assert v0 > 0.35, (
    f"CHI {v0:.3f} ~ NGAU NHIEN -> HF da KHOI TAO LAI dau phan loai thay vi nap dau da huan luyen. "
    "Mo hinh mat sach kha nang xep hang; loss van se giam dep nhung ket qua se te. "
    "DAY RAT CO THE LA NGUYEN NHAN LUOT FINE-TUNE CU RA -6,50 (luu tru: 'probe - dau hoc lai: -6,34'). "
    "Sua: kiem config num_labels cua model goc, nap bang AutoModelForSequenceClassification KHONG truyen num_labels.")
model.train(); t0=time.time(); losses=[]
for step,(enc,g,k) in enumerate(dl,1):
    with torch.autocast("cuda",dtype=torch.float16):
        logits = model(**enc).logits.view(g,k)
        loss = Fn.cross_entropy(logits, torch.zeros(g,dtype=torch.long,device="cuda"))/ACC
    scaler.scale(loss).backward()
    losses.append(loss.item()*ACC)
    if step%ACC==0:
        scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step()
    if step==1:
        torch.cuda.synchronize()
        pk=torch.cuda.max_memory_allocated()/2**30
        tot=torch.cuda.get_device_properties(0).total_memory/2**30
        print(f"  buoc 1 OK · dinh VRAM {pk:.2f}/{tot:.2f} GB · du dia {tot-pk:.2f} GB", flush=True)
        if tot-pk < 1.0: print("  !!! DU DIA DUOI 1 GB -> se OOM o cau co doan dai. Ha MAXLEN xuong 768.")
    if step%100==0:
        el=(time.time()-t0)/60
        print(f"  buoc {step}/{total} · loss {np.mean(losses[-100:]):.4f} · {el:.1f} phut · con ~{el/step*(total-step):.0f} phut", flush=True)
    if step==500:
        d=np.mean(losses[:100])-np.mean(losses[-100:])
        v5=val_acc()
        print(f"  >>> CHOT 500 BUOC: loss giam {d:+.4f} · kiem tra {v5:.4f} (truoc {v0:.4f})")
        # 10-09 SUA: KHONG duoc gate bang LOSS o moc nay.
        #   ACC=16 -> 500 buoc = ~31 buoc toi uu. OneCycleLR pct_start=0.1 tren
        #   total_steps=5107//16+1=320 => WARMUP = 32 buoc toi uu = 512 buoc batch.
        #   Nen 100 buoc DAU chay o LR~0 (model gan nhu khong doi, loss = loss ban dau)
        #   va 100 buoc CUOI o LR~max (model dang di, loss cao hon + nhieu hon).
        #   Loss tang o cua so nay la DUNG THEO THIET KE, khong phai dau hieu hong.
        #   Lan chay 10-09 da bi DUNG OAN o day (d=-0.1208) trong khi val acc
        #   tang 0.7300 -> 0.7650. Gate bang VAL ACC.
        if v5 <= v0: print(f"  !!! VAL ACC KHONG TANG ({v0:.4f} -> {v5:.4f}) -> DUNG."); break
print(f"\nxong · kiem tra SAU huan luyen: {val_acc():.4f}")
model.save_pretrained("/kaggle/working/ft_listwise"); tok.save_pretrained("/kaggle/working/ft_listwise")
print("da luu /kaggle/working/ft_listwise — TAI VE TRUOC KHI DONG PHIEN")

In [ ]:
# ===== DANH GIA THAT: xep lai top-5 cua dev300 bang mo hinh moi =====
# DC, CTX, DEVF, SCORESF da giai quyet o o 1 (hong thi hong tu giay thu 15)
dev=json.load(open(DEVF,encoding="utf-8"))
S=json.load(open(SCORESF,encoding="utf-8"))
gold={q:{str(x) for x in v["answer"]} for q,v in dev.items()}; Q=list(gold)
mx=lambda v: max(v["ce"], v.get("ce_deep",-9e9))
order={q:[d for d,_ in sorted(S[q].items(), key=lambda kv:-mx(kv[1]))] for q in Q}
base=np.mean([order[q][0] in gold[q] for q in Q])
assert abs(base-0.7100)<1e-6, "SAI FILE DIEM"
print(f"moc = {base:.4f} · tran top-5 = {np.mean([bool(gold[q]&set(order[q][:5])) for q in Q]):.4f}")

K_VIEW=3
# GIAI PHONG trang thai huan luyen TRUOC khi nap ban sao thu hai cua model.
# Khong lam -> AdamW (~6,8 GB) + 2 model (~4,6 GB) = OOM o gio thu 4.
try:
    del opt, scaler, dl, sched
except NameError:
    pass
import gc; gc.collect(); torch.cuda.empty_cache()
model.gradient_checkpointing_disable()     # suy luan: tat cho nhanh
print(f"VRAM sau khi don: {torch.cuda.memory_allocated()/2**30:.2f} GB")

# ---- DOI CHUNG BAT BUOC: mo hinh GOC tren DUNG harness nay (3 doan, 1 goc nhin) ----
# Moc 0.7100 la cua max(ce,ce_deep): HAI goc nhin, toi 50 doan. So voi no la so tao voi cam.
# headview_run da do mo hinh goc tren harness nay = 0.6033. Do lai tai cho cho chac.
from transformers import AutoModelForSequenceClassification as AMSC
base_model = AMSC.from_pretrained(MODEL).cuda().eval()

def rerank(m):
    out={}
    with torch.no_grad():
        for i,q in enumerate(Q,1):
            qt=dev[q]["question"]; best,bs=None,-9e9
            for d in order[q][:5]:
                ck=DC.pick_chunks(qt,CTX,d,k=K_VIEW)
                e=tok([qt]*len(ck), ck, truncation=True, max_length=MAXLEN, padding=True, return_tensors="pt")
                with torch.autocast("cuda",dtype=torch.float16):
                    s=float(m(**{k:v.cuda() for k,v in e.items()}).logits.max())
                if s>bs: bs,best=s,d
            out[q]=best
            if i%150==0: print(f"    {i}/{len(Q)}", flush=True)
    return out

print("cham nhanh DOI CHUNG (mo hinh goc)...")
ctrl = rerank(base_model)
CTRL = np.mean([ctrl[q] in gold[q] for q in Q])
print(f"DOI CHUNG: mo hinh GOC tren harness nay = {CTRL:.4f}   <-- DAY moi la moc dung")
del base_model; torch.cuda.empty_cache()

print("cham mo hinh FINE-TUNE...")
model.eval(); new={}
new = rerank(model)

from math import comb
def mc(A,B):
    w=int((A&~B).sum()); l=int((B&~A).sum()); n=w+l
    p=(sum(comb(n,i) for i in range(max(w,l),n+1))/2**n*2) if n else 1.0
    return w,l,min(p,1.0)
A=np.array([new[q] in gold[q] for q in Q])
C=np.array([ctrl[q] in gold[q] for q in Q])
got=A.mean(); w,l,pv=mc(A,C)
print(f"\n{'='*66}")
print(f"  mo hinh GOC  (harness 3 doan) = {CTRL:.4f}   <-- doi chung")
print(f"  mo hinh FT   (harness 3 doan) = {got:.4f}   ({got-CTRL:+.4f})")
print(f"  McNemar FT vs GOC: thang {w} thua {l}  p={pv:.4f}")
print(f"  (tham chieu: san xuat max(ce,ce_deep) 2 goc nhin/50 doan = {base:.4f} · tran top-5 = 0.9333)")
print('='*66)
d = got-CTRL
if   d >= 0.05: print("HUAN LUYEN AN RO -> chay lai ca pipeline (tang1+tang2) bang mo hinh FT, roi nop.")
elif d >= 0.02: print("CO TIN HIEU -> chay lai pipeline day du de biet no co vuot 0.7100 khong.")
elif d > -0.02: print("HOA -> huan luyen khong them gi. Giu mo hinh goc.")
else:           print("HONG -> dung. Dung van tham so cuu (bai hoc 2 lan truoc).")
